このコードは、政府調達ポータルの「落札実績オープンデータのダウンロード」  https://www.p-portal.go.jp/pps-web-biz/UAB02/OAB0201でダウンロードしたcsv形式ファイルを、左ウィンドウにアップロードし、データ整形するコードです。（2025年06月更新）

In [ ]:
# 手動アップロード版

In [ ]:
from datetime import datetime
import pandas as pd
import os
import requests
import io
import zipfile

# --- (1) MAPPING DEFINITIONS (省略 - 変更なし) ---
MINISTRY_AGENCY_MAPPING = {
    # ... 府省庁マッピング ...
    "衆議院": "A1", "参議院": "B1", "国立国会図書館": "C1", "最高裁判所": "D1",
    "会計検査院": "E1", "人事院": "F1", "国家公務員倫理審査会": "F2", "内閣官房": "G1",
    "内閣法制局": "H1", "安全保障会議": "I1", "内閣府": "J1", "宮内庁": "J2",
    "公正取引委員会": "J3", "国家公安委員会": "J4", "警察庁": "J5", "金融庁": "J6",
    "消費者庁": "J7", "個人情報保護委員会": "J8", "J9": "カジノ管理委員会", "総務省": "K1",
    "公害等調整委員会": "K2", "消防庁": "K3", "法務省": "L1", "L2": "検察庁",
    "公安審査委員会": "L3", "公安調査庁": "L4", "外務省": "M1", "財務省": "N1",
    "国税庁": "N2", "文部科学省": "O1", "文化庁": "O2", "スポーツ庁": "O3",
    "厚生労働省": "P1", "中央労働委員会": "P2", "農林水産省": "Q1", "林野庁": "Q2",
    "水産庁": "Q3", "経済産業省": "R1", "資源エネルギー庁": "R2", "特許庁": "R3",
    "中小企業庁": "R4", "国土交通省": "S1", "運輸安全委員会": "S2", "観光庁": "S3",
    "気象庁": "S4", "海上保安庁": "S5", "環境省": "T1", "原子力安全庁": "T2",
    "防衛省": "U1", "復興庁": "V1", "デジタル庁": "W1", "こども家庭庁": "JA",
    "福岡SMC": "S1", "東北地方整備局": "S1", "北海道開発局": "S1", "近畿地方整備局": "S1",
    "関東地方整備局": "S1", "九州地方整備局": "S1", "中国地方整備局": "S1", "四国地方整備局": "S1",
    "中部地方整備局": "S1", "北陸地方整備局": "S1", "防衛装備庁": "U1", "地方防衛局": "U1",
    "防衛医科大学校": "U1", "防衛監察本部": "U1", "防衛大学校": "U1", "防衛省共済組合": "U1",
    "航空自衛隊": "U1", "海上自衛隊": "U1", "陸上自衛隊": "U1", "国立研究開発法人理化学研究所": "O1",
    "国立研究開発法人宇宙航空研究開発機構": "O1", "国立研究開発法人産業技術総合研究所": "R1",
    "国立研究開発法人農業・食品産業技術総合研究機構": "Q1",
    "国立研究開発法人水産研究・教育機構": "Q1", "独立行政法人国立印刷局": "N1", "独立行政法人造幣局": "N1",
    "独立行政法人国立病院機構": "P1", "独立行政法人地域医療機能推進機構": "P1",
    "独立行政法人労働者健康安全機構": "P1", "独立行政法人国立高等専門学校機構": "O1",
    "独立行政法人大学改革支援・学位授与機構": "O1", "独立行政法人国立青少年教育振興機構": "O1",
    "独立行政法人国立女性教育会館": "O1", "独立行政法人日本スポーツ振興センター": "O3",
    "独立行政法人国立科学博物館": "O2", "独立行政法人国立美術館": "O2", "独立行政法人国立文化財機構": "O2",
    "独立行政法人日本芸術文化振興会": "O2", "独立行政法人日本学術振興会": "O1",
    "独立行政法人日本学生支援機構": "O1", "独立行政法人教職員支援機構": "O1",
    "独立行政法人科学技術振興機構": "O1", "独立行政法人物質・材料研究機構": "O1",
    "独立行政法人防災科学技術研究所": "O1", "独立行政法人量子科学技術研究開発機構": "O1",
    "独立行政法人国立医薬品食品衛生研究所": "P1", "独立行政法人医薬基盤・健康・栄養研究所": "P1",
    "独立行政法人医薬品医療機器総合機構": "P1", "独立行政法人福祉医療機構": "P1",
    "独立行政法人高齢・障害・求職者雇用支援機構": "P1", "独立行政法人勤労者退職金共済機構": "P1",
    "独立行政法人奄美群島振興開発基金": "S1", "独立行政法人水資源機構": "S1",
    "独立行政法人鉄道建設・運輸施設整備支援機構": "S1", "独立行政法人国際観光振興機構": "S3",
    "独立行政法人自動車事故対策機構": "S1", "独立行政法人海上災害防止センター": "S5",
    "独立行政法人航空保安大学校": "S1", "独立行政法人国立環境研究所": "T1",
    "独立行政法人環境再生保全機構": "T1", "独立行政法人製品評価技術基盤機構": "R1",
    "独立行政法人中小企業基盤整備機構": "R4", "独立行政法人情報処理推進機構": "W1",
    "国立研究開発法人国立がん研究センター": "P1", "国立研究開発法人国立循環器病研究センター": "P1",
    "国立研究開発法人国立精神・神経医療研究センター": "P1", "国立研究開発法人国立国際医療研究センター": "P1",
    "国立研究開発法人国立成育医療研究センター": "P1", "国立研究開発法人国立長寿医療研究センター": "P1",
    "国立研究開発法人日本医療研究開発機構": "P1",
    "A1": "A1", "B1": "B1", "C1": "C1", "D1": "D1", "E1": "E1", "F1": "F1", "F2": "F2",
    "G1": "G1", "H1": "H1", "I1": "I1", "J1": "J1", "J2": "J2", "J3": "J3", "J4": "J4",
    "J5": "J5", "J6": "J6", "J7": "J7", "J8": "J8", "J9": "J9", "K1": "K1", "K2": "K2",
    "K3": "K3", "L1": "L1", "L2": "L2", "L3": "L3", "L4": "L4", "M1": "M1", "N1": "N1",
    "N2": "N2", "O1": "O1", "O2": "O2", "O3": "O3", "P1": "P1", "P2": "P2", "Q1": "Q1",
    "Q2": "Q2", "Q3": "Q3", "R1": "R1", "R2": "R2", "R3": "R3", "R4": "R4", "S1": "S1",
    "S2": "S2", "S3": "S3", "S4": "S4", "S5": "S5", "T1": "T1", "T2": "T2", "U1": "U1",
    "V1": "V1", "W1": "W1", "JA": "JA"
}

MINISTRY_CODE_TO_NAME = {v: k for k, v in MINISTRY_AGENCY_MAPPING.items()}
MINISTRY_CODE_TO_NAME.update({
    "A1": "衆議院", "B1": "参議院", "C1": "国立国会図書館", "D1": "最高裁判所",
    "E1": "会計検査院", "F1": "人事院", "F2": "国家公務員倫理審査会", "G1": "内閣官房",
    "H1": "内閣法制局", "I1": "安全保障会議", "J1": "内閣府", "J2": "宮内庁",
    "J3": "公正取引委員会", "J4": "国家公安委員会", "J5": "警察庁", "J6": "金融庁",
    "J7": "消費者庁", "J8": "個人情報保護委員会", "J9": "カジノ管理委員会", "K1": "総務省",
    "K2": "公害等調整委員会", "K3": "消防庁", "L1": "法務省", "L2": "検察庁",
    "L3": "公安審査委員会", "L4": "公安調査庁", "M1": "外務省", "N1": "財務省",
    "N2": "国税庁", "O1": "文部科学省", "O2": "文化庁", "O3": "スポーツ庁",
    "P1": "厚生労働省", "P2": "中央労働委員会", "Q1": "農林水産省", "Q2": "林野庁",
    "Q3": "水産庁", "R1": "経済産業省", "R2": "資源エネルギー庁", "R3": "特許庁",
    "R4": "中小企業庁", "S1": "国土交通省", "S2": "運輸安全委員会", "S3": "観光庁",
    "S4": "気象庁", "S5": "海上保安庁", "T1": "環境省", "T2": "原子力安全庁",
    "U1": "防衛省", "V1": "復興庁", "W1": "デジタル庁", "JA": "こども家庭庁"
})

BIDDING_METHOD_MAPPING = {
    "8002010": "一般競争入札・最低価格", "8002020": "一般競争入札・最高価格",
    "8002040": "一般競争入札・総合評価", "8002050": "一般競争入札・複数落札",
    "8003010": "指名競争入札・最低価格", "8003020": "指名競争入札・最高価格",
    "8003040": "指名競争入札・総合評価", "8003050": "指名競争入札・複数落札",
    "8004025": "随意契約方式・複数業者", "8001010": "随意契約方式・オープンカウンタ",
    "8004020": "随意契約方式・特定業者", "8004030": "随意契約方式・公募型プロポーザル方式",
    "8014025": "随意契約方式・複数業者・少額", "8011010": "随意契約方式・オープンカウンタ・少額",
    "8014020": "随意契約方式・特定業者・少額", "8014030": "随意契約方式・公募型プロポーザル方式・少額",
}

# --- (2) HELPER FUNCTIONS (省略 - 変更なし) ---
def get_ministry_agency_code(raw_ministry_data, is_older_format=False):
    if pd.isna(raw_ministry_data):
        return None
    raw_ministry_data_str = str(raw_ministry_data).strip()
    if is_older_format:
        return MINISTRY_AGENCY_MAPPING.get(raw_ministry_data_str, None)
    else:
        if raw_ministry_data_str in MINISTRY_AGENCY_MAPPING:
            return MINISTRY_AGENCY_MAPPING[raw_ministry_data_str]
        best_match_code = None
        best_match_len = 0
        for name_key, code in MINISTRY_AGENCY_MAPPING.items():
            if name_key in raw_ministry_data_str:
                if len(name_key) > best_match_len:
                    best_match_len = len(name_key)
                    best_match_code = code
        return best_match_code

def get_bidding_method_name(bidding_method_code):
    return BIDDING_METHOD_MAPPING.get(str(bidding_method_code), None)


# --- (3) NEW FUNCTION FOR AUTOMATED PROCESSING (変更なし) ---
def process_zip_data_from_urls(base_url, years):
    all_dfs = []

    COLUMNS_7_COLS = ['ID', '調達件名／府省庁名', '契約日', '落札金額（単位：円）',
                      '入札方式コード', '事業者名', '参照ID']
    COLUMNS_8_COLS = ['ID', '調達件名', '契約日', '落札金額（単位：円）',
                      '府省コード_raw', '入札方式コード', '事業者名', '参照ID']

    for year in years:
        zip_filename = f'successful_bid_record_info_all_{year}.zip'
        zip_url = f'{base_url}{zip_filename}'
        print(f"\nProcessing '{zip_filename}' from {zip_url}...")

        try:
            # 1. Download the ZIP file content
            response = requests.get(zip_url, stream=True, timeout=30)
            response.raise_for_status()

            # 2. Open the ZIP file in memory (io.BytesIO)
            with io.BytesIO(response.content) as zip_buffer:
                with zipfile.ZipFile(zip_buffer) as zf:
                    csv_name = next((name for name in zf.namelist() if name.endswith('.csv')), None)

                    if not csv_name:
                        print(f"  Warning: No CSV file found inside {zip_filename}. Skipping.")
                        continue

                    # 3. Read the CSV file content from memory
                    with zf.open(csv_name) as csv_file:
                        csv_data = io.TextIOWrapper(csv_file, encoding='utf-8-sig')
                        df = pd.read_csv(csv_data, on_bad_lines='skip', header=None)

            print(f"  Loaded {len(df)} rows from '{csv_name}' (inside zip).")

            # 4. Determine file format and apply columns
            num_cols = len(df.columns)
            is_older_format = False

            if num_cols == len(COLUMNS_7_COLS):
                df.columns = COLUMNS_7_COLS
                print(f"  Detected 7-column format.")
            elif num_cols == len(COLUMNS_8_COLS):
                df.columns = COLUMNS_8_COLS
                is_older_format = True
                print(f"  Detected 8-column (older) format.")
            else:
                print(f"  Error: Unexpected number of columns ({num_cols}). Skipping this file.")
                continue

            # 5. Apply the ministry/agency code and bidding method logic
            if is_older_format:
                df['府省コード'] = df['府省コード_raw'].apply(lambda x: get_ministry_agency_code(x, is_older_format=True))
            else:
                df['府省コード'] = df['調達件名／府省庁名'].apply(lambda x: get_ministry_agency_code(x, is_older_format=False))
                df['調達件名'] = df['調達件名／府省庁名']
                df = df.drop(columns=['調達件名／府省庁名'])

            df['入札方式名称'] = df['入札方式コード'].astype(str).apply(get_bidding_method_name)

            # 6. Extract Year
            df['契約日'] = pd.to_datetime(df['契約日'], errors='coerce')
            df['Year'] = df['契約日'].dt.year

            all_dfs.append(df)

        except requests.exceptions.RequestException as req_e:
            print(f"  Error downloading {zip_filename} (HTTP/Network error): {req_e}. Skipping.")
        except zipfile.BadZipFile:
            print(f"  Error: {zip_filename} is a corrupted or invalid ZIP file. Skipping.")
        except Exception as e:
            print(f"  An unexpected error occurred while processing {zip_filename}: {e}. Skipping.")

    if not all_dfs:
        print("No valid dataframes were loaded. Exiting.")
        return

    # --- (4) CONCATENATION, DEDUPLICATION, AND FINAL OUTPUT (省略 - 変更なし) ---
    final_columns_order_before_ministry_name = [
        'ID', '調達件名', '契約日', '落札金額（単位：円）',
        '事業者名', '参照ID', '入札方式コード', '入札方式名称', '府省コード', 'Year'
    ]

    print("\nConcatenating all loaded DataFrames...")
    combined_df = pd.concat(all_dfs, ignore_index=True)
    combined_df = combined_df.reindex(columns=final_columns_order_before_ministry_name)
    print(f"Total rows in combined DataFrame: {len(combined_df)}")

    initial_rows = len(combined_df)
    print(f"Checking for duplicate rows (initial count: {initial_rows})...")
    combined_df.drop_duplicates(inplace=True)
    rows_after_dedup = len(combined_df)
    duplicates_removed = initial_rows - rows_after_dedup
    print(f"Duplicate rows removed: {duplicates_removed}")
    print(f"Rows after removing duplicates: {rows_after_dedup}")

    print("\nAdding '省庁名' (Ministry/Agency Name) column...")
    combined_df['省庁名'] = combined_df['府省コード'].map(MINISTRY_CODE_TO_NAME)

    final_columns_order_with_ministry_name = [
        'ID', '調達件名', '契約日', '落札金額（単位：円）',
        '事業者名', '参照ID', '入札方式コード', '入札方式名称',
        '府省コード', '省庁名', 'Year'
    ]
    combined_df = combined_df.reindex(columns=final_columns_order_with_ministry_name)

    timestamp = datetime.now().strftime("%Y-%m%d-%H%M")
    output_csv_path = f'integrated_pubbid_data_{timestamp}.csv'

    print("\nProcessed Data head (after deduplication and adding 省庁名):")
    print(combined_df.head())

    combined_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
    print(f"\nIntegrated data saved to {output_csv_path}")

    # --- (5) VALUE COUNTS TABLE (省略 - 変更なし) ---
    print("\n--- Value counts for '入札方式名称' by Year and Total ---")
    years = sorted(combined_df['Year'].dropna().unique().astype(int).tolist())
    annual_counts_dfs = []

    for year in years:
        df_year = combined_df[combined_df['Year'] == year]
        if not df_year.empty:
            annual_vc = df_year['入札方式名称'].value_counts().rename(f'{year}')
            annual_counts_dfs.append(annual_vc)
        else:
            print(f"No data for year {year}")

    total_vc = combined_df['入札方式名称'].value_counts().rename('Total')
    annual_counts_dfs.append(total_vc)

    if annual_counts_dfs:
        combined_vc_df = pd.concat(annual_counts_dfs, axis=1).fillna(0).astype(int)

        headers = ['入札方式名称'] + combined_vc_df.columns.tolist()
        column_widths = [len(str(header)) for header in headers]

        for index, row in combined_vc_df.iterrows():
            column_widths[0] = max(column_widths[0], len(str(index)))
            for i, col in enumerate(combined_vc_df.columns):
                column_widths[i+1] = max(column_widths[i+1], len(str(row[col])))

        header_separator = '+' + '+'.join(['-' * (width + 2) for width in column_widths]) + '+'
        row_separator = '+' + '+'.join(['-' * (width + 2) for width in column_widths]) + '+'

        table_output = [header_separator]
        header_row_str = '| ' + ' | '.join([f"{headers[i]:<{column_widths[i]}}" if i == 0 else f"{headers[i]:>{column_widths[i]}}" for i in range(len(headers))]) + ' |'
        table_output.append(header_row_str)
        table_output.append(row_separator)

        for index, row in combined_vc_df.iterrows():
            row_values = [str(index)] + [str(row[col]) for col in combined_vc_df.columns]
            row_str = '| ' + ' | '.join([f"{row_values[i]:<{column_widths[i]}}" if i == 0 else f"{row_values[i]:>{column_widths[i]}}" for i in range(len(row_values))]) + ' |'
            table_output.append(row_str)

        table_output.append(header_separator)

        print('\n'.join(table_output))
    else:
        print("No '入札方式名称' data available to display value counts.")

# --- (6) EXECUTION BLOCK ---
if __name__ == "__main__":
    # !!! 実行前に、[実際のデータ公開ドメインとパス]を正しいURLに置き換えてください !!!
    # 例: "https://example.com/data/download/"
    BASE_URL = "http://[実際のデータ公開ドメイン]/[パス]/"
    TARGET_YEARS = [2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017, 2016, 2015, 2014, 2013]

    print("Starting automated ZIP download and processing...")
    process_zip_data_from_urls(BASE_URL, TARGET_YEARS)

Starting automated ZIP download and processing...

Processing 'successful_bid_record_info_all_2025.zip' from http://[実際のデータ公開ドメイン]/[パス]/successful_bid_record_info_all_2025.zip...
  Error downloading successful_bid_record_info_all_2025.zip (HTTP/Network error): Failed to parse: http://[実際のデータ公開ドメイン]/[パス]/successful_bid_record_info_all_2025.zip. Skipping.

Processing 'successful_bid_record_info_all_2024.zip' from http://[実際のデータ公開ドメイン]/[パス]/successful_bid_record_info_all_2024.zip...
  Error downloading successful_bid_record_info_all_2024.zip (HTTP/Network error): Failed to parse: http://[実際のデータ公開ドメイン]/[パス]/successful_bid_record_info_all_2024.zip. Skipping.

Processing 'successful_bid_record_info_all_2023.zip' from http://[実際のデータ公開ドメイン]/[パス]/successful_bid_record_info_all_2023.zip...
  Error downloading successful_bid_record_info_all_2023.zip (HTTP/Network error): Failed to parse: http://[実際のデータ公開ドメイン]/[パス]/successful_bid_record_info_all_2023.zip. Skipping.

Processing 'successful_bid_reco

In [ ]:
from datetime import datetime
import pandas as pd
import os
import re # For regular expressions to parse filenames

# Define the mappings for 府省コード and 入札方式コード (unchanged)
MINISTRY_AGENCY_MAPPING = {
    "衆議院": "A1", "参議院": "B1", "国立国会図書館": "C1", "最高裁判所": "D1",
    "会計検査院": "E1", "人事院": "F1", "国家公務員倫理審査会": "F2", "内閣官房": "G1",
    "内閣法制局": "H1", "安全保障会議": "I1", "内閣府": "J1", "宮内庁": "J2",
    "公正取引委員会": "J3", "国家公安委員会": "J4", "警察庁": "J5", "金融庁": "J6",
    "消費者庁": "J7", "個人情報保護委員会": "J8", "カジノ管理委員会": "J9", "総務省": "K1",
    "公害等調整委員会": "K2", "消防庁": "K3", "法務省": "L1", "検察庁": "L2",
    "公安審査委員会": "L3", "公安調査庁": "L4", "外務省": "M1", "財務省": "N1",
    "国税庁": "N2", "文部科学省": "O1", "文化庁": "O2", "スポーツ庁": "O3",
    "厚生労働省": "P1", "中央労働委員会": "P2", "農林水産省": "Q1", "林野庁": "Q2",
    "水産庁": "Q3", "経済産業省": "R1", "資源エネルギー庁": "R2", "特許庁": "R3",
    "中小企業庁": "R4", "国土交通省": "S1", "運輸安全委員会": "S2", "観光庁": "S3",
    "気象庁": "S4", "海上保安庁": "S5", "環境省": "T1", "原子力安全庁": "T2",
    "防衛省": "U1", "復興庁": "V1", "デジタル庁": "W1", "こども家庭庁": "JA",
    "福岡SMC": "S1", "東北地方整備局": "S1", "北海道開発局": "S1", "近畿地方整備局": "S1",
    "関東地方整備局": "S1", "九州地方整備局": "S1", "中国地方整備局": "S1",
    "四国地方整備局": "S1", "中部地方整備局": "S1", "北陸地方整備局": "S1",
    "防衛装備庁": "U1", "地方防衛局": "U1", "防衛医科大学校": "U1", "防衛監察本部": "U1",
    "防衛大学校": "U1", "防衛省共済組合": "U1", "航空自衛隊": "U1", "海上自衛隊": "U1",
    "陸上自衛隊": "U1",
    "国立研究開発法人理化学研究所": "O1", "国立研究開発法人宇宙航空研究開発機構": "O1",
    "国立研究開発法人産業技術総合研究所": "R1",
    "国立研究開発法人農業・食品産業技術総合研究機構": "Q1", "国立研究開発法人水産研究・教育機構": "Q1",
    "独立行政法人国立印刷局": "N1", "独立行政法人造幣局": "N1",
    "独立行政法人国立病院機構": "P1", "独立行政法人地域医療機能推進機構": "P1",
    "独立行政法人労働者健康安全機構": "P1", "独立行政法人国立高等専門学校機構": "O1",
    "独立行政法人大学改革支援・学位授与機構": "O1", "独立行政法人国立青少年教育振興機構": "O1",
    "独立行政法人国立女性教育会館": "O1", "独立行政法人日本スポーツ振興センター": "O3",
    "独立行政法人国立科学博物館": "O2", "独立行政法人国立美術館": "O2",
    "独立行政法人国立文化財機構": "O2", "独立行政法人日本芸術文化振興会": "O2",
    "独立行政法人日本学術振興会": "O1", "独立行政法人日本学生支援機構": "O1",
    "独立行政法人教職員支援機構": "O1", "独立行政法人科学技術振興機構": "O1",
    "独立行政法人物質・材料研究機構": "O1", "独立行政法人防災科学技術研究所": "O1",
    "独立行政法人量子科学技術研究開発機構": "O1", "独立行政法人国立医薬品食品衛生研究所": "P1",
    "独立行政法人医薬基盤・健康・栄養研究所": "P1", "独立行政法人医薬品医療機器総合機構": "P1",
    "独立行政法人福祉医療機構": "P1", "独立行政法人高齢・障害・求職者雇用支援機構": "P1",
    "独立行政法人勤労者退職金共済機構": "P1",
    "独立行政法人奄美群島振興開発基金": "S1",
    "独立行政法人水資源機構": "S1",
    "独立行政法人鉄道建設・運輸施設整備支援機構": "S1",
    "独立行政法人国際観光振興機構": "S3",
    "独立行政法人自動車事故対策機構": "S1",
    "独立行政法人海上災害防止センター": "S5",
    "独立行政法人航空保安大学校": "S1",
    "独立行政法人国立環境研究所": "T1",
    "独立行政法人環境再生保全機構": "T1",
    "独立行政法人製品評価技術基盤機構": "R1",
    "独立行政法人中小企業基盤整備機構": "R4",
    "独立行政法人情報処理推進機構": "W1",
    "国立研究開発法人国立がん研究センター": "P1",
    "国立研究開発法人国立循環器病研究センター": "P1",
    "国立研究開発法人国立精神・神経医療研究センター": "P1",
    "国立研究開発法人国立国際医療研究センター": "P1",
    "国立研究開発法人国立成育医療研究センター": "P1",
    "国立研究開発法人国立長寿医療研究センター": "P1",
    "国立研究開発法人日本医療研究開発機構": "P1",
    # Codes for direct lookup (for older files where code is in a dedicated column)
    "A1": "A1", "B1": "B1", "C1": "C1", "D1": "D1", "E1": "E1", "F1": "F1", "F2": "F2",
    "G1": "G1", "H1": "H1", "I1": "I1", "J1": "J1", "J2": "J2", "J3": "J3", "J4": "J4",
    "J5": "J5", "J6": "J6", "J7": "J7", "J8": "J8", "J9": "J9", "K1": "K1", "K2": "K2",
    "K3": "K3", "L1": "L1", "L2": "L2", "L3": "L3", "L4": "L4", "M1": "M1", "N1": "N1",
    "N2": "N2", "O1": "O1", "O2": "O2", "O3": "O3", "P1": "P1", "P2": "P2", "Q1": "Q1",
    "Q2": "Q2", "Q3": "Q3", "R1": "R1", "R2": "R2", "R3": "R3", "R4": "R4", "S1": "S1",
    "S2": "S2", "S3": "S3", "S4": "S4", "S5": "S5", "T1": "T1", "T2": "T2", "U1": "U1",
    "V1": "V1", "W1": "W1", "JA": "JA"
}

# Invert MINISTRY_AGENCY_MAPPING to get code -> name mapping for the new column
MINISTRY_CODE_TO_NAME = {v: k for k, v in MINISTRY_AGENCY_MAPPING.items()}
MINISTRY_CODE_TO_NAME.update({
    "A1": "衆議院", "B1": "参議院", "C1": "国立国会図書館", "D1": "最高裁判所",
    "E1": "会計検査院", "F1": "人事院", "F2": "国家公務員倫理審査会", "G1": "内閣官房",
    "H1": "内閣法制局", "I1": "安全保障会議", "J1": "内閣府", "J2": "宮内庁",
    "J3": "公正取引委員会", "J4": "国家公安委員会", "J5": "警察庁", "J6": "金融庁",
    "J7": "消費者庁", "J8": "個人情報保護委員会", "J9": "カジノ管理委員会", "K1": "総務省",
    "K2": "公害等調整委員会", "K3": "消防庁", "L1": "法務省", "L2": "検察庁",
    "L3": "公安審査委員会", "L4": "公安調査庁", "M1": "外務省", "N1": "財務省",
    "N2": "国税庁", "O1": "文部科学省", "O2": "文化庁", "O3": "スポーツ庁",
    "P1": "厚生労働省", "P2": "中央労働委員会", "Q1": "農林水産省", "Q2": "林野庁",
    "Q3": "水産庁", "R1": "経済産業省", "R2": "資源エネルギー庁", "R3": "特許庁",
    "R4": "中小企業庁", "S1": "国土交通省", "S2": "運輸安全委員会", "S3": "観光庁",
    "S4": "気象庁", "S5": "海上保安庁", "T1": "環境省", "T2": "原子力安全庁",
    "U1": "防衛省", "V1": "復興庁", "W1": "デジタル庁", "JA": "こども家庭庁"
})


BIDDING_METHOD_MAPPING = {
    "8002010": "一般競争入札・最低価格", "8002020": "一般競争入札・最高価格",
    "8002040": "一般競争入札・総合評価", "8002050": "一般競争入札・複数落札",
    "8003010": "指名競争入札・最低価格", "8003020": "指名競争入札・最高価格",
    "8003040": "指名競争入札・総合評価", "8003050": "指名競争入札・複数落札",
    "8004025": "随意契約方式・複数業者", "8001010": "随意契約方式・オープンカウンタ",
    "8004020": "随意契約方式・特定業者", "8004030": "随意契約方式・公募型プロポーザル方式",
    "8014025": "随意契約方式・複数業者・少額", "8011010": "随意契約方式・オープンカウンタ・少額",
    "8014020": "随意契約方式・特定業者・少額", "8014030": "随意契約方式・公募型プロポーザル方式・少額",
}

def get_ministry_agency_code(raw_ministry_data, is_older_format=False):
    """
    Looks up the ministry/agency code based on the raw data.
    If is_older_format is True, it expects the raw data to be a code.
    Otherwise, it expects a name string and performs partial matching.
    """
    if pd.isna(raw_ministry_data):
        return None

    raw_ministry_data_str = str(raw_ministry_data).strip()

    if is_older_format:
        # For older format, the column directly contains the code (e.g., 'S1')
        return MINISTRY_AGENCY_MAPPING.get(raw_ministry_data_str, None)
    else:
        # For newer format, the column contains the name (e.g., '福岡SMC（九州南部）技：')
        # Try direct match first if the key is the full string
        if raw_ministry_data_str in MINISTRY_AGENCY_MAPPING:
            return MINISTRY_AGENCY_MAPPING[raw_ministry_data_str]

        # Search for partial matches, prioritizing longer matches to avoid mis-mapping
        best_match_code = None
        best_match_len = 0
        for name_key, code in MINISTRY_AGENCY_MAPPING.items():
            if name_key in raw_ministry_data_str:
                if len(name_key) > best_match_len:
                    best_match_len = len(name_key)
                    best_match_code = code
        return best_match_code

def get_bidding_method_name(bidding_method_code):
    """
    Looks up the bidding method name based on the code.
    Ensures the code is treated as a string for lookup.
    """
    return BIDDING_METHOD_MAPPING.get(str(bidding_method_code), None)

def find_latest_integrated_csv(directory="/content/"):
    """
    Finds the latest 'integrated_pubbid_data_YYYY-MMDD-HHMM.csv' file in the given directory.
    """
    pattern = re.compile(r'integrated_pubbid_data_(\d{4}-\d{2}\d{2}-\d{4})\.csv')
    latest_file = None
    latest_timestamp = None

    if not os.path.exists(directory):
        print(f"Error: Directory '{directory}' does not exist.")
        return None

    for f_name in os.listdir(directory):
        match = pattern.fullmatch(f_name)
        if match:
            timestamp_str = match.group(1)
            try:
                # Convert timestamp string to datetime object for comparison
                file_timestamp = datetime.strptime(timestamp_str, "%Y-%m%d-%H%M")
                if latest_timestamp is None or file_timestamp > latest_timestamp:
                    latest_timestamp = file_timestamp
                    latest_file = f_name
            except ValueError:
                # Handle cases where timestamp might be malformed in filename
                continue
    return latest_file


def format_table_output(df_to_print, title, index_col_name, num_cols_right_align):
    """
    Formats a DataFrame into a string with ASCII borders and prints it.
    Args:
        df_to_print (pd.DataFrame): The DataFrame to format and print.
        title (str): Title for the table.
        index_col_name (str): The name of the column that will be used as the index (left-aligned).
        num_cols_right_align (int): The number of right-aligned columns (from the right).
    """
    if df_to_print.empty:
        print(f"{title}\n該当するデータがありません。")
        return

    # Prepare DataFrame for printing: set index for the first column
    df_formatted = df_to_print.set_index(index_col_name)

    # Get column headers
    headers = [index_col_name] + df_formatted.columns.tolist()

    # Calculate max width for each column
    column_widths = [len(str(header)) for header in headers]

    # Update width for the index column
    column_widths[0] = max(column_widths[0], df_formatted.index.map(lambda x: len(str(x))).max())

    # Update widths for data columns
    for i, col in enumerate(df_formatted.columns):
        column_widths[i+1] = max(column_widths[i+1], df_formatted[col].astype(str).map(len).max())

    # Create separator lines
    header_separator = '+' + '+'.join(['-' * (width + 2) for width in column_widths]) + '+'
    row_separator = '+' + '+'.join(['-' * (width + 2) for width in column_widths]) + '+'

    # Build table output
    table_output = [f"\n{title}\n", header_separator]

    # Header row
    header_row_parts = []
    for i in range(len(headers)):
        if i == 0:
            header_row_parts.append(f"{headers[i]:<{column_widths[i]}}") # Left align index header
        else:
            header_row_parts.append(f"{headers[i]:>{column_widths[i]}}") # Right align other headers
    table_output.append('| ' + ' | '.join(header_row_parts) + ' |')
    table_output.append(row_separator)

    # Data rows
    for idx, row in df_formatted.iterrows():
        row_values = [str(idx)] + [str(row[col]) for col in df_formatted.columns]
        row_parts = []
        for i in range(len(row_values)):
            if i == 0:
                row_parts.append(f"{row_values[i]:<{column_widths[i]}}") # Left align index values
            else:
                row_parts.append(f"{row_values[i]:>{column_widths[i]}}") # Right align data values
        table_output.append('| ' + ' | '.join(row_parts) + ' |')

    table_output.append(header_separator) # Bottom border

    print('\n'.join(table_output))


def analyze_bidding_data(directory="/content/"):
    """
    Loads the latest integrated CSV, and analyzes bidding methods and contractors.
    """
    latest_csv_file = find_latest_integrated_csv(directory)

    if not latest_csv_file:
        print(f"Error: No 'integrated_pubbid_data_YYYY-MMDD-HHMM.csv' found in {directory}.")
        print("Please ensure the previous step (data integration) was successful and the file exists.")
        return

    full_csv_path = os.path.join(directory, latest_csv_file)
    print(f"Loading latest integrated data from: {full_csv_path}")

    try:
        df = pd.read_csv(full_csv_path, encoding='utf-8-sig')
        print(f"Successfully loaded {len(df)} rows.")
    except Exception as e:
        print(f"Error loading {full_csv_path}: {e}")
        return

    # Ensure '落札金額（単位：円）' is numeric
    df['落札金額（単位：円）'] = pd.to_numeric(df['落札金額（単位：円）'], errors='coerce')
    # Drop rows where '落札金額（単位：円）' could not be converted (NaNs) or is negative
    df.dropna(subset=['落札金額（単位：円）'], inplace=True)
    df = df[df['落札金額（単位：円）'] >= 0] # Exclude negative or invalid amounts

    # Ensure '事業者名' is string type and handle potential empty/NaN strings
    df['事業者名'] = df['事業者名'].astype(str).replace('nan', '不明な事業者').str.strip()
    df = df[df['事業者名'] != '不明な事業者'] # Exclude unknown contractors

    print("\n--- 分析：入札種類別 受注額・受注件数トップ50事業者 ---")

    # Group by Bidding Method and Contractor Name
    grouped_by_bidding_and_contractor = df.groupby(['入札方式名称', '事業者名']).agg(
        受注額=('落札金額（単位：円）', 'sum'),
        受注件数=('ID', 'count')
    ).reset_index()

    # Get unique bidding methods to iterate through
    bidding_methods = grouped_by_bidding_and_contractor['入札方式名称'].unique()

    for method in sorted(bidding_methods):
        method_df = grouped_by_bidding_and_contractor[grouped_by_bidding_and_contractor['入札方式名称'] == method].copy() # Use .copy() to avoid SettingWithCopyWarning

        # Format 受注額 for display before sorting, to calculate correct column widths
        method_df['受注額_表示用'] = method_df['受注額'].apply(lambda x: f"{x:,.0f} 円")

        # Top 50 by 受注額
        top_by_amount = method_df.sort_values(by='受注額', ascending=False).head(50)
        format_table_output(
            top_by_amount[['事業者名', '受注額_表示用', '受注件数']],
            f"==== 入札方式名称: {method} - 受注額 トップ50 ====",
            '事業者名',
            2 # '受注額_表示用' and '受注件数' are right-aligned
        )

        # Top 50 by 受注件数
        top_by_count = method_df.sort_values(by='受注件数', ascending=False).head(50)
        format_table_output(
            top_by_count[['事業者名', '受注件数', '受注額_表示用']],
            f"==== 入札方式名称: {method} - 受注件数 トップ50 ====",
            '事業者名',
            2 # '受注件数' and '受注額_表示用' are right-aligned
        )

# Run the analysis
if __name__ == "__main__":
    analyze_bidding_data()

Loading latest integrated data from: /content/integrated_pubbid_data_2025-0618-0446.csv
Successfully loaded 254679 rows.

--- 分析：入札種類別 受注額・受注件数トップ50事業者 ---

==== 入札方式名称: 一般競争入札・最低価格 - 受注額 トップ50 ====

+---------------------+-------------------+------+
| 事業者名                |           受注額_表示用 | 受注件数 |
+---------------------+-------------------+------+
| 日本電気株式会社            | 112,487,904,375 円 | 1014 |
| 沖電気工業株式会社           |  54,154,123,864 円 |  298 |
| 東芝インフラシステムズ株式会社     |  50,093,415,000 円 |  345 |
| 株式会社神戸製鋼所           |  33,838,730,000 円 |    6 |
| 東京電力エナジーパートナー株式会社   |  30,980,834,689 円 |  282 |
| 富士通株式会社             |  27,861,612,075 円 |  577 |
| ゼロワットパワー株式会社        |  26,237,310,743 円 |  606 |
| 日本無線株式会社            |  25,980,653,250 円 |  631 |
| 一般財団法人航空保安協会        |  24,259,699,010 円 |  116 |
| 株式会社ケーネス            |  22,015,720,000 円 |  576 |
| アズビル株式会社            |  21,504,086,800 円 |  386 |
| 首都圏ビルサービス協同組合       |  21,050,728,655 円 |  160 |
| 株式会社エヌ・ティ・ティ・データ    |  19,098,954

▽入札種類別（入札方式名称別）の受注額と受注件数のトップ50表示

・最新の CSV ファイルの自動検出: /content/ ディレクトリ内の integrated_pubbid_data_YYYY-MMDD-HHMM.csv パターンに一致するファイルの中から、最も新しいタイムスタンプを持つファイルを自動的に見つけて読み込みます。
・入札方式名称別のトップ50表示:
入札方式名称 ごとにデータをグループ化します。
各グループ内で、事業者名 ごとに 落札金額（単位：円） の合計 (受注額) と ID のカウント (受注件数) を計算します。
計算された 受注額 と 受注件数 に基づいて、それぞれトップ50の事業者を表示します。